# UP03: Targeting policy with a stated cost

Question: which fraction of customers should be treated to maximize net gain, when each treatment costs `c` in conversion units?

Data: Criteo uplift dataset from the Kaggle slug `arashnic/uplift-modeling`, read in place from `/kaggle/input/uplift-modeling/`. Split is random and stratified by treatment (train 50%, validation 20%, test 30%), seed 42. The cutoff is chosen on validation and reported once on test.

Run with **Run All** inside a Kaggle Notebook with the dataset attached. This notebook was not executed locally because the data is not on this machine.

In [ ]:
import glob
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from uplift_policy.data import FEATURES, OUTCOME, TREATMENT, load_criteo, three_way_split
from uplift_policy.metrics import qini_coefficient, qini_curve, uplift_by_decile
from uplift_policy.model import TLearner
from uplift_policy.policy import apply_cutoff, choose_cutoff, cumulative_net_gain, evaluate_policy

RANDOM_STATE = 42
SAMPLE_N = None  # e.g. 1_000_000 for a faster first run; None uses all rows
COST = 0.01  # assumed cost per treated customer, in conversion units (see README limitation)
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

csv_files = glob.glob("/kaggle/input/uplift-modeling/*.csv")
assert csv_files, "Attach the arashnic/uplift-modeling dataset to this notebook."
df = load_criteo(csv_files[0], sample_n=SAMPLE_N)
train, val, test = three_way_split(df)
print({name: len(part) for name, part in [("train", train), ("validation", val), ("test", test)]})

## Fit the T-learner on train and score validation and test

In [ ]:
model = TLearner().fit(train[FEATURES], train[TREATMENT], train[OUTCOME])
val_score = model.predict_uplift(val[FEATURES])
test_score = model.predict_uplift(test[FEATURES])

## Choose the cutoff on validation

In [ ]:
best = choose_cutoff(val[OUTCOME], val[TREATMENT], val_score, cost=COST)
print(best)

val_table = cumulative_net_gain(val[OUTCOME], val[TREATMENT], val_score, cost=COST)
fig, ax = plt.subplots(figsize=(7, 4))
finite = val_table[np.isfinite(val_table["net_gain"])]
ax.plot(finite["fraction"], finite["net_gain"], color="tab:blue")
ax.axvline(best["fraction"], color="tab:red", linestyle="--", label=f"chosen share {best['fraction']:.2f}")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Treated share")
ax.set_ylabel("Expected net gain (conversions)")
ax.set_title(f"Criteo uplift, validation split, T-learner, cost={COST}")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "up03_treated_share_vs_gain.png", dpi=150)
plt.show()

## Report once on test: Qini, uplift by decile, and the chosen policy against random targeting

In [ ]:
y_test, t_test = test[OUTCOME].to_numpy(), test[TREATMENT].to_numpy()

qini = qini_coefficient(y_test, t_test, test_score)
print(f"Qini coefficient, T-learner on test: {qini:.4f} (random targeting: 0 by construction)")

mask = apply_cutoff(test_score, best["cutoff_score"])
policy = evaluate_policy(y_test, t_test, mask, cost=COST)

# Random targeting at the same treated share, using the overall test uplift.
ate = y_test[t_test == 1].mean() - y_test[t_test == 0].mean()
n_sel = policy["n_selected"]
random_net = n_sel * (ate - COST)
summary = pd.DataFrame(
    [
        {"policy": "T-learner cutoff", "treated_share": n_sel / len(test), "uplift_selected": policy["uplift_selected"], "net_gain": policy["net_gain"]},
        {"policy": "random, same share", "treated_share": n_sel / len(test), "uplift_selected": ate, "net_gain": random_net},
    ]
)
summary

In [ ]:
x, gain = qini_curve(y_test, t_test, test_score)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x, gain, label="T-learner")
ax.plot([0, 1], [0, gain[-1]], linestyle="--", color="grey", label="random")
ax.set_xlabel("Fraction of population targeted")
ax.set_ylabel("Incremental conversions")
ax.set_title("Criteo uplift, test split, T-learner Qini curve")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "up03_qini_curve.png", dpi=150)
plt.show()

In [ ]:
deciles = uplift_by_decile(y_test, t_test, test_score)
display(deciles)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["tab:blue" if r else "lightgrey" for r in deciles["reliable"]]
ax.bar(deciles["bin"].astype(str), deciles["uplift"], color=colors)
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Score decile (1 = highest)")
ax.set_ylabel("Treated rate minus control rate")
ax.set_title("Criteo uplift, test split, T-learner uplift by decile (grey = fewer than 30 per arm)")
fig.tight_layout()
fig.savefig(FIG_DIR / "up03_uplift_by_decile.png", dpi=150)
plt.show()